In [32]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.output_parsers import DatetimeOutputParser

from config import OPEN_AI_KEY as API_KEY

In [11]:
chat = ChatOpenAI(model="gpt-5.6-luna", seed=365, temperature=0, api_key=API_KEY, max_completion_tokens=300)
msg_human = HumanMessage("Can you give me an interesting fact I probably didn't know about?")

In [10]:
response = chat.invoke([msg_human])
response.content

'The Eiffel Tower can become **about 15 centimeters taller in summer**. Its iron expands as it heats up, and the tower can also subtly tilt toward the warmer, sunlit side.'

In [35]:
str_output_parser = StrOutputParser()
response_parsed = str_output_parser.invoke(response)
response_parsed

ValidationError: 1 validation error for Generation
text
  Input should be a valid string [type=string_type, input_value=[AIMessage(content='Congr... 0, 'reasoning': 129}})], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

In [12]:
# Without format instructions
msg_human = HumanMessage("Could you suggest some dog names?")
response = chat.invoke([msg_human])
print(response.content)

Absolutely! Here are some dog-name ideas by style:

**Classic:**  
- Max  
- Bella  
- Charlie  
- Daisy  
- Cooper  
- Lucy  

**Cute:**  
- Mochi  
- Biscuit  
- Waffles  
- Pickle  
- Noodle  
- Pudding  

**Nature-inspired:**  
- Willow  
- River  
- Maple  
- Clover  
- Aspen  
- Bear  

**Unique:**  
- Juniper  
- Ziggy  
- Cosmo  
- Miso  
- Atlas  
- Echo  

**Strong:**  
- Thor  
- Scout  
- Nova  
- Loki  
- Blaze  
- Ranger  

**For a small dog:**  
- Peanut  
- Pip  
- Button  
- Tater Tot  
- Bean  
- Sprout  

If you tell me the dog’s breed, color, personality, or gender, I can suggest names that fit especially well.


In [3]:
list_parser = CommaSeparatedListOutputParser()
instructions = list_parser.get_format_instructions()
print(instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [18]:
# With format instructions
msg_human = HumanMessage(f"Could you suggest some dog names? {instructions}")
response = chat.invoke([msg_human])
print(response.content)

Luna, Milo, Daisy, Max, Bella, Charlie, Coco, Bailey, Scout, Teddy, Ruby, Winston, Nala, Finn, Olive, Archie, Willow, Rocky, Penny, Jasper


In [21]:
response_parsed = list_parser.invoke(response)
print(response_parsed)


['Luna', 'Milo', 'Daisy', 'Max', 'Bella', 'Charlie', 'Coco', 'Bailey', 'Scout', 'Teddy', 'Ruby', 'Winston', 'Nala', 'Finn', 'Olive', 'Archie', 'Willow', 'Rocky', 'Penny', 'Jasper']


### Date Parser

In [26]:
date_parser = DatetimeOutputParser()
instructions = date_parser.get_format_instructions()
print(instructions)

Write a datetime string that matches the following pattern: '%Y-%m-%dT%H:%M:%S.%fZ'.

Examples: 2023-07-04T14:30:00.000000Z, 1999-12-31T23:59:59.999999Z, 2025-01-01T00:00:00.000000Z

Return ONLY this string, no other words!


In [27]:
# Without format instructions
msg_human = HumanMessage("When was Tinubu born?")
response = chat.invoke([msg_human])
print(response.content)

Bola Ahmed Tinubu, Nigeria’s president, was born on **29 March 1952**, according to his official records.


In [28]:
msg_human = HumanMessage(f"When was Tinubu born? {instructions}")
response = chat.invoke([msg_human])
print(response.content)

1952-03-29T00:00:00.000000Z


In [29]:
response_parsed = date_parser.invoke(response)
print(response_parsed)

1952-03-29 00:00:00


### Piping a model, prompt and output parser

In [5]:
list_instructions = list_parser.get_format_instructions()
chat_template = ChatPromptTemplate.from_messages([
    ("human", "I've recently adopted a {pet}. Could you suggest three {pet} names? \n" + list_instructions)
])
chat_template

ChatPromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template="I've recently adopted a {pet}. Could you suggest three {pet} names? \nYour response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`"), additional_kwargs={})])

In [12]:
chat_template_result = chat_template.invoke({"pet": "dog"})
chat_result = chat.invoke(chat_template_result)

In [13]:
chat_result

AIMessage(content='Luna, Milo, Daisy', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 48, 'total_tokens': 57, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'compute_units': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EJyceyt6rrjTYq6LcLoC8xA4bhOVY', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a066bb-a886-7cb1-a503-37acf936394b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 9, 'total_tokens': 57, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [14]:
response_parsed = list_parser.invoke(chat_result)
print(response_parsed)

['Luna', 'Milo', 'Daisy']


### Batching

In [23]:
%%time
chat = ChatOpenAI(model="gpt-5.6-luna", seed=365, temperature=0, api_key=API_KEY)
chat_template = ChatPromptTemplate.from_messages([
    ("human", "I just adopted a {pet} which is a {breed}. Can you suggest training tips?")
])
chain = chat_template | chat
response = chain.invoke({"pet": "dog", "breed": "shepard"})
response

CPU times: user 12.1 ms, sys: 469 μs, total: 12.6 ms
Wall time: 9.5 s


AIMessage(content='Congratulations on your new dog! Shepherds—such as German, Australian, or Belgian Shepherds—are often intelligent, energetic, sensitive, and eager to work. These tips can help:\n\n### 1. Start with a quiet adjustment period\nGive your dog a predictable routine and a calm space of their own. Avoid overwhelming them with visitors, dog parks, or intense outings during the first several days or weeks.\n\n### 2. Use positive reinforcement\nReward behaviors you want with small treats, praise, toys, or access to something enjoyable. Keep sessions short—about 5–10 minutes—and end on a success. Avoid yelling or punishment, which can increase fear or reactivity.\n\n### 3. Teach the basics\nPrioritize:\n- Their name and attention (“look at me”)\n- Sit, down, stay, and come\n- Leave it and drop it\n- Loose-leash walking\n- Calm greetings\n- Going to a bed or mat\n\nPractice in a quiet room first, then gradually add distractions.\n\n### 4. Provide daily mental and physical activi

In [24]:
%%time
response = chain.batch([{"pet": "dog", "breed": "shepard"}, {"pet": "dragon", "breed": "night fury"}])
response

CPU times: user 14.9 ms, sys: 0 ns, total: 14.9 ms
Wall time: 11.7 s


[AIMessage(content='Congratulations on your new dog! Shepherds are often intelligent, energetic, loyal, and sensitive, so they usually do best with structure, mental stimulation, and reward-based training.\n\n### Start with adjustment\n- Give your dog a quiet “home base” with a bed, water, and safe chew toys.\n- Keep the first days calm; avoid overwhelming them with visitors, dog parks, or long outings.\n- Establish consistent routines for meals, walks, sleep, and bathroom breaks.\n- Schedule a veterinary checkup, especially to discuss vaccinations, microchipping, diet, and any fear or health concerns.\n\n### Use positive reinforcement\n- Reward desired behavior immediately with treats, praise, toys, or access to something enjoyable.\n- Keep sessions short—about 5–10 minutes—several times a day.\n- Avoid yelling, hitting, leash corrections, or “alpha” techniques. These can increase fear and defensive behavior, especially in a newly adopted dog.\n\n### Teach the basics\nBegin with:\n- *

In [50]:
pass_through = RunnablePassthrough()
msg_tool = "What are the five most important tool a {job} needs? Answer only by listing the tools."
msg_strategy = "Considering the tools provided, develop a strategy (not more than 300 words) for effectively learning and mastering them: {tools}"
chat_template_tools = ChatPromptTemplate.from_template(msg_tool)
chat_template_strategy = ChatPromptTemplate.from_template(msg_strategy)

chain_tools = chat_template_tools | chat | str_output_parser
chain_strategy = chat_template_strategy | chat | str_output_parser

In [39]:
result = chain_tools.invoke({"job": "data scientist"})
print(print(type(result)))

1. Python
2. SQL
3. Jupyter Notebook
4. Git
5. Tableau


In [41]:
result_strategy = chain_strategy.invoke({"tools": result})
result_strategy


'## Recommended learning strategy\n\nLearn the tools as one integrated data-analysis workflow rather than as isolated subjects:\n\n> **Git → Python → Jupyter → SQL → Tableau → integrated projects**\n\nIn practice, Git should be used from the beginning, while Python, SQL, Jupyter, and Tableau are developed together.\n\n---\n\n## 1. Establish the foundation: Git and environment setup\n\n### Goals\n- Understand version control and reproducible work.\n- Become comfortable working from a terminal or command prompt.\n- Create a consistent project structure.\n\n### Learn\n- Repositories, commits, branches, merges, and remotes\n- `clone`, `add`, `commit`, `push`, `pull`, `status`, `log`\n- `.gitignore`\n- Branch-based development\n- README files and meaningful commit messages\n\n### Practice\nCreate a repository such as:\n\n```text\ndata-analysis-project/\n├── data/\n├── notebooks/\n├── scripts/\n├── sql/\n├── dashboards/\n├── README.md\n└── requirements.txt\n```\n\nCommit your work frequently

In [45]:
print(result_strategy)

## Recommended learning strategy

Learn the tools as one integrated data-analysis workflow rather than as isolated subjects:

> **Git → Python → Jupyter → SQL → Tableau → integrated projects**

In practice, Git should be used from the beginning, while Python, SQL, Jupyter, and Tableau are developed together.

---

## 1. Establish the foundation: Git and environment setup

### Goals
- Understand version control and reproducible work.
- Become comfortable working from a terminal or command prompt.
- Create a consistent project structure.

### Learn
- Repositories, commits, branches, merges, and remotes
- `clone`, `add`, `commit`, `push`, `pull`, `status`, `log`
- `.gitignore`
- Branch-based development
- README files and meaningful commit messages

### Practice
Create a repository such as:

```text
data-analysis-project/
├── data/
├── notebooks/
├── scripts/
├── sql/
├── dashboards/
├── README.md
└── requirements.txt
```

Commit your work frequently. Do not wait until a project is finish

In [46]:
result_strategy

'## Recommended learning strategy\n\nLearn the tools as one integrated data-analysis workflow rather than as isolated subjects:\n\n> **Git → Python → Jupyter → SQL → Tableau → integrated projects**\n\nIn practice, Git should be used from the beginning, while Python, SQL, Jupyter, and Tableau are developed together.\n\n---\n\n## 1. Establish the foundation: Git and environment setup\n\n### Goals\n- Understand version control and reproducible work.\n- Become comfortable working from a terminal or command prompt.\n- Create a consistent project structure.\n\n### Learn\n- Repositories, commits, branches, merges, and remotes\n- `clone`, `add`, `commit`, `push`, `pull`, `status`, `log`\n- `.gitignore`\n- Branch-based development\n- README files and meaningful commit messages\n\n### Practice\nCreate a repository such as:\n\n```text\ndata-analysis-project/\n├── data/\n├── notebooks/\n├── scripts/\n├── sql/\n├── dashboards/\n├── README.md\n└── requirements.txt\n```\n\nCommit your work frequently

In [47]:
runnable_chain_tools = chat_template_tools | chat | str_output_parser | {"tools": pass_through}
runnable_chain_tools

ChatPromptTemplate(input_variables=['job'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job'], input_types={}, partial_variables={}, template='What are the five most important tool a {job} needs? Answer only by listing the tools.'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-5.6 Luna', 'release_date': '2026-07-09', 'last_updated': '2026-07-09', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_tool_message': True, 'image

In [48]:
runnable_result = runnable_chain_tools.invoke({"job": "Python Django"})
print(runnable_result)

{'tools': '1. Python  \n2. Django  \n3. pip  \n4. Virtual environment  \n5. Git'}


In [51]:
chain_combined = runnable_chain_tools | chain_strategy
runnable_result_2 = chain_combined.invoke({"job": "Python Django"})
print(runnable_result_2)

## Strategy for Learning and Mastering the Tools

### 1. Start with Python Fundamentals
Learn variables, data types, control flow, functions, modules, exceptions, file handling, object-oriented programming, and testing. Practice daily by solving small problems and building command-line projects such as a calculator, task manager, or file organizer.

### 2. Learn Git from the Beginning
Create a Git repository for every project. Master:

- `git init`, `clone`, `add`, `commit`, `status`, and `log`
- Branching, merging, and resolving conflicts
- Remote repositories using GitHub or GitLab
- Writing meaningful commit messages

Use Git to track progress rather than waiting until projects become complex.

### 3. Master Virtual Environments and pip
For each Python project, create and activate an isolated environment:

```bash
python -m venv .venv
```

Install dependencies with pip:

```bash
pip install package-name
pip freeze > requirements.txt
pip install -r requirements.txt
```

Understand pa